In [1]:
import sys
from pathlib import Path
repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from PIL import Image as PILImageModule
from PIL.Image import Image
import numpy as np
import numpy.typing as npt
from transformers import pipeline
import json

SAMPLE_IDX = 0
results_dir = f"../results/new_object_placement/fixed/{SAMPLE_IDX}"

background_image = PILImageModule.open(f"../datasets/csc2529/background/{SAMPLE_IDX}.png")
result_image = PILImageModule.open(results_dir+"/result.png")
object_mask = PILImageModule.open(results_dir+"/debug/object_mask.png")
object_mask_np = np.asarray(object_mask).astype(np.bool_)
target_bbox_mask = PILImageModule.open(results_dir+"/debug/target_bbox_mask.png")

def points_to_array(json: dict[str, int]) -> tuple[npt.NDArray[np.intp], npt.NDArray[np.intp]]:
    """Parses json dict of source pixel and target pixel to to numpy arrays."""
    return (
        np.array([json["source_x"], json["source_y"]], dtype=np.intp),
        np.array([json["target_x"], json["target_y"]], dtype=np.intp),
    )

def get_source_target_px() -> tuple[npt.NDArray[np.intp], npt.NDArray[np.intp]]:
    with open("../datasets/csc2529/depth.json") as depth_file:
        depth_json = json.load(depth_file)
        for idx, entry in depth_json.items():
            if idx == SAMPLE_IDX:
                return points_to_array(entry)
    return np.array([]), np.array([])

source_px, target_px = get_source_target_px()

/mnt/perma/bin/miniconda3/envs/depthedit/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
depth_pipeline = pipeline(
    task="depth-estimation", model="LiheYoung/depth-anything-small-hf"
)
result_depth_image: Image = depth_pipeline(  # type: ignore
    result_image
)["depth"]
result_depth_np = np.asarray(result_depth_image)
print(result_depth_np.max())

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


255


In [3]:
sam_pipeline = pipeline("mask-generation", model="facebook/sam-vit-huge", device=0)
depth_pipeline = pipeline(
    task="depth-estimation", model="LiheYoung/depth-anything-small-hf"
)

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [4]:
from utils.metrics import calc_metrics


metrics = calc_metrics(
    background_image=np.asarray(background_image),
    result_image=np.asarray(result_image),
    source_mask=object_mask_np,
    target_bbox_mask=np.asarray(target_bbox_mask),
    scale_applied=3.0,
    timings={},
    sam_pipeline=sam_pipeline,
    depth_pipeline=depth_pipeline,
)


In [5]:
print(metrics)

{'z_mean_before': 0.016250939028316494, 'z_std_before': 0.11405024462999655, 'z_mean_after': 0.0065428638595296336, 'z_std_after': 0.022546740093972673, 'z_mean_delta': -0.00970807516878686, 'depth_ratio': 2.48376542401186, 'area_before': 2120.0, 'area_after': 22210.0, 'scale_applied': 3.0, 'scale_observed': 3.2367290733608867, 'scale_error': 0.7529636493490268, 'scale_error_rel': 0.30315409099012863, 'fid': 97.90120603669453, 'ssim': 0.7638372832949232, 'mask_iou': 0.6558269739617287}


In [6]:
from utils.img_proc import get_object_mask


def get_mask_depth(
    depth_map: npt.NDArray[np.float32], mask: npt.NDArray[np.bool_]
) -> tuple[float, float]:
    """Returns the mean and std of all the depth values inside the mask."""

    masked_disparity = depth_map[mask]
    if masked_disparity.size == 0:
        return float("nan"), float("nan")
    if masked_disparity.mean() == 0 or masked_disparity.std() == 0:
        return float("nan"), float("nan")
    mean_depth = 1.0 / float(masked_disparity.mean())
    std_depth = 1.0 / float(masked_disparity.std())
    return mean_depth, std_depth


x, y = np.where(np.array(target_bbox_mask, dtype=np.bool_))
target_center_px = np.floor(np.array(list(zip(y, x))).mean(axis=0)).astype(np.intp)
target_mask: npt.NDArray[np.bool_] = get_object_mask(
    np.asarray(result_image), target_center_px, sam_pipeline
)

print(get_mask_depth(result_depth_np, object_mask_np))
print(get_mask_depth(result_depth_np, target_mask))

(0.020523941371231633, 0.07821634146754544)
(0.0065428638595296336, 0.022546740093972673)
